# Startup Simulator - AI Training

Train AI agents for the Startup Simulator board game.

**Two training modes:**
- **Single Agent** - Train one agent via frozen-pool self-play (fast, good for 2-player)
- **Tournament** - Train 4 agents in round-robin competition (slower, stronger agents)

**Steps:**
1. Setup (clone + install)
2. Choose training mode and run
3. View training progress
4. Export ONNX model for web app
5. Download

## 1. Setup

In [ ]:
!git clone https://github.com/ue-too/startup-tabletop.git
%cd startup-tabletop

# Install with compatible versions for Colab Python 3.12
!pip install "numpy<2" gymnasium==0.29.1 -q
!pip install -e ".[dev]" sb3-contrib onnx onnxruntime matplotlib -q

# Suppress gym deprecation warning
import warnings
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# Verify
import torch
torch.distributions.Distribution.set_default_validate_args(False)
from startup_simulator.engine import GameEngine
engine = GameEngine(num_players=2, seed=42)
print(f'Engine OK: {len(engine.get_legal_actions())} legal actions')
print(f'GPU available: {torch.cuda.is_available()}')

---
## 2A. Single Agent Training (Frozen-Pool Self-Play)

Best for 2-player games. Fast (~2k steps/sec).

| Steps | Time | Expected win rate vs random |
|-------|------|----------------------------|
| 1M | ~8 min | ~95% |
| 2M | ~16 min | ~98% |
| 5M | ~40 min | ~100% |

In [ ]:
import os, time, json
import torch
torch.distributions.Distribution.set_default_validate_args(False)

from sb3_contrib import MaskablePPO
from training.frozen_pool_env import FrozenPoolEnv
from training.callbacks import SelfPlayCallback

# === CONFIGURATION ===
TIMESTEPS = 2_000_000
POOL_INTERVAL = 200_000
# =====================

pool_dir = 'checkpoints/pool'
save_dir = 'checkpoints/model'
os.makedirs(pool_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

# Seed pool
if not any(f.endswith('.zip') for f in os.listdir(pool_dir)):
    env_tmp = FrozenPoolEnv(seed=0, pool_dir=pool_dir)
    MaskablePPO('MultiInputPolicy', env_tmp, verbose=0).save(os.path.join(pool_dir, 'pool_000.zip'))
    del env_tmp

env = FrozenPoolEnv(seed=42, reward_mode='shaped', pool_dir=pool_dir)
model = MaskablePPO(
    'MultiInputPolicy', env,
    n_steps=512, batch_size=128, n_epochs=4,
    learning_rate=3e-4, gamma=0.99, ent_coef=0.01,
    max_grad_norm=0.5, verbose=0,
)

# Track progress
progress = {'steps': [], 'scores': [], 'win_rates': [], 'episodes': []}
callback = SelfPlayCallback(save_dir=save_dir, save_freq=POOL_INTERVAL, eval_freq=POOL_INTERVAL, eval_games=30, verbose=1)

print(f'Training {TIMESTEPS:,} steps...')
start = time.time()
trained = 0
pool_counter = 1

while trained < TIMESTEPS:
    chunk = min(POOL_INTERVAL, TIMESTEPS - trained)
    model.learn(total_timesteps=chunk, callback=callback, reset_num_timesteps=False)
    trained += chunk
    model.save(os.path.join(pool_dir, f'pool_{pool_counter:03d}_{trained//1000}k.zip'))
    pool_counter += 1
    
    # Quick eval for progress tracking
    from training.evaluate import evaluate_agents, make_random_agent, make_sb3_agent
    trained_fn = make_sb3_agent(os.path.join(save_dir, 'model_final'))
    r = evaluate_agents([trained_fn, make_random_agent(42)], num_games=50)
    wr = r['win_rates'][0]
    avg = r['avg_scores'][0]
    progress['steps'].append(trained)
    progress['win_rates'].append(wr)
    progress['scores'].append(avg)
    progress['episodes'].append(callback._episode_scores.__len__())
    
    elapsed = time.time() - start
    print(f'  [{trained//1000}k] vs_random={wr:.0%}, avg_score={avg:.1f}, {elapsed:.0f}s')

model.save(os.path.join(save_dir, 'model_final'))
print(f'Done in {time.time()-start:.0f}s')

# Save progress
with open(os.path.join(save_dir, 'progress.json'), 'w') as f:
    json.dump(progress, f)

### Single Agent Progress

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

steps_k = [s/1000 for s in progress['steps']]

# Win rate
ax1.plot(steps_k, [w*100 for w in progress['win_rates']], 'b-o', linewidth=2, markersize=6)
ax1.set_xlabel('Training Steps (k)')
ax1.set_ylabel('Win Rate vs Random (%)')
ax1.set_title('Win Rate Progress')
ax1.set_ylim(0, 105)
ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Average score
ax2.plot(steps_k, progress['scores'], 'g-o', linewidth=2, markersize=6)
ax2.set_xlabel('Training Steps (k)')
ax2.set_ylabel('Average Score')
ax2.set_title('Score Progress')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Break even')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to training_progress.png')

---
## 2B. Tournament Training (4-Agent Round-Robin)

Train 4 distinct agents that compete against each other. Each round, every agent trains against the other 3.

| Rounds | Steps/Round | Total Steps | Time | Description |
|--------|------------|------------|------|-------------|
| 5 | 100k | 2M | ~20 min | Quick tournament |
| 10 | 200k | 8M | ~80 min | Standard |
| 20 | 200k | 16M | ~160 min | Strong agents |

In [ ]:
import os, time, json
import torch
torch.distributions.Distribution.set_default_validate_args(False)

from training.train_tournament import train_tournament

# === CONFIGURATION ===
ROUNDS = 10
STEPS_PER_ROUND = 200_000
BASE_DIR = 'checkpoints/tournament'
# =====================

history = train_tournament(
    rounds=ROUNDS,
    steps_per_round=STEPS_PER_ROUND,
    base_dir=BASE_DIR,
    threads=2,  # Colab has 2 CPU cores
    eval_games=50,
    verbose=True,
)

### Tournament Progress

In [ ]:
import matplotlib.pyplot as plt
import json

# Load history
history_path = 'checkpoints/tournament/training_history.json'
with open(history_path) as f:
    history = json.load(f)

rounds_data = history['rounds']
num_agents = history['config']['num_agents']
round_nums = [r['round'] for r in rounds_data]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['#00bcd4', '#ffc107', '#4caf50', '#9c27b0']
agent_names = [f'Agent {i}' for i in range(num_agents)]

# 1. Win rate vs Random
ax = axes[0][0]
for aid in range(num_agents):
    wr = [r['eval'].get(f'Agent_{aid}_vs_random', {}).get('win_rate', 0.5) for r in rounds_data]
    ax.plot(round_nums, [w*100 for w in wr], '-o', color=colors[aid], label=agent_names[aid], linewidth=2, markersize=5)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Round')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate vs Random')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

# 2. Win rate vs Heuristic
ax = axes[0][1]
for aid in range(num_agents):
    wr = [r['eval'].get(f'Agent_{aid}_vs_heuristic', {}).get('win_rate', 0.5) for r in rounds_data]
    ax.plot(round_nums, [w*100 for w in wr], '-o', color=colors[aid], label=agent_names[aid], linewidth=2, markersize=5)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Round')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate vs Heuristic')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)

# 3. Average score vs Random
ax = axes[1][0]
for aid in range(num_agents):
    scores = [r['eval'].get(f'Agent_{aid}_vs_random', {}).get('avg_score', 0) for r in rounds_data]
    ax.plot(round_nums, scores, '-o', color=colors[aid], label=agent_names[aid], linewidth=2, markersize=5)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Round')
ax.set_ylabel('Average Score')
ax.set_title('Score vs Random')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Head-to-head heatmap (latest round)
ax = axes[1][1]
latest = rounds_data[-1]['eval']
matrix = np.zeros((num_agents, num_agents))
for i in range(num_agents):
    for j in range(num_agents):
        if i == j:
            matrix[i][j] = 0.5
        elif i < j:
            h2h = latest.get(f'Agent_{i}_vs_Agent_{j}', {})
            matrix[i][j] = h2h.get('a_win_rate', 0.5)
            matrix[j][i] = h2h.get('b_win_rate', 0.5)

im = ax.imshow(matrix * 100, cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(num_agents))
ax.set_yticks(range(num_agents))
ax.set_xticklabels(agent_names)
ax.set_yticklabels(agent_names)
ax.set_title(f'Head-to-Head (Round {rounds_data[-1]["round"]})')
for i in range(num_agents):
    for j in range(num_agents):
        ax.text(j, i, f'{matrix[i][j]*100:.0f}%', ha='center', va='center', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, label='Win Rate %')

plt.tight_layout()
plt.savefig('tournament_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to tournament_progress.png')

---
## 3. Export Best Agent to ONNX

In [ ]:
import torch, numpy as np, os, glob
torch.distributions.Distribution.set_default_validate_args(False)
from sb3_contrib import MaskablePPO

# Find the best model - either from single agent or tournament
model_path = None

# Check single agent
if os.path.exists('checkpoints/model/model_final.zip'):
    model_path = 'checkpoints/model/model_final'
    print(f'Using single-agent model: {model_path}')

# Check tournament (pick agent with highest win rate vs random)
if os.path.exists('checkpoints/tournament/training_history.json'):
    import json
    with open('checkpoints/tournament/training_history.json') as f:
        hist = json.load(f)
    if hist['rounds']:
        latest_eval = hist['rounds'][-1]['eval']
        best_agent = max(range(4), key=lambda i: latest_eval.get(f'Agent_{i}_vs_random', {}).get('win_rate', 0))
        agent_dir = f'checkpoints/tournament/agent_{best_agent}'
        zips = sorted(glob.glob(f'{agent_dir}/*.zip'))
        if zips:
            model_path = zips[-1].replace('.zip', '')
            print(f'Using tournament Agent {best_agent}: {model_path}')

if model_path is None:
    print('ERROR: No trained model found. Run training first.')
else:
    model = MaskablePPO.load(model_path)
    policy = model.policy
    OBS_SIZE, MAX_ACTIONS = 2800, 512

    class PolicyWrapper(torch.nn.Module):
        def __init__(self, p):
            super().__init__()
            self.fe = p.features_extractor
            self.mlp = p.mlp_extractor
            self.act = p.action_net
        def forward(self, obs, mask):
            f = self.fe({'observation': obs, 'action_mask': mask})
            pi, _ = self.mlp(f)
            logits = self.act(pi)
            return logits + (mask - 1.0) * 1e8

    wrapper = PolicyWrapper(policy)
    wrapper.eval()
    dummy_obs = torch.zeros(1, OBS_SIZE)
    dummy_mask = torch.ones(1, MAX_ACTIONS)

    output_path = 'ai_model.onnx'
    torch.onnx.export(
        wrapper, (dummy_obs, dummy_mask), output_path,
        input_names=['observation', 'action_mask'],
        output_names=['logits'],
        dynamic_axes={'observation': {0: 'b'}, 'action_mask': {0: 'b'}, 'logits': {0: 'b'}},
        opset_version=17,
    )

    import onnxruntime as ort
    session = ort.InferenceSession(output_path)
    ort_out = session.run(None, {'observation': dummy_obs.numpy(), 'action_mask': dummy_mask.numpy()})
    with torch.no_grad():
        pt_out = wrapper(dummy_obs, dummy_mask)
    diff = np.abs(pt_out.numpy() - ort_out[0]).max()
    size_mb = os.path.getsize(output_path) / (1024*1024)
    print(f'\nONNX exported: {size_mb:.1f} MB, max diff: {diff:.6f}')
    print(f'Ready to download!')

## 4. Download

In [ ]:
from google.colab import files

# Download ONNX model (for web app)
files.download('ai_model.onnx')

# Download training progress charts
for f in ['training_progress.png', 'tournament_progress.png']:
    if os.path.exists(f):
        files.download(f)

## 5. Resume Training (Optional)

Upload a previous checkpoint to continue training:

In [ ]:
# # Upload previous model
# from google.colab import files
# uploaded = files.upload()
# 
# # For single agent:
# import shutil
# shutil.move('model_final.zip', 'checkpoints/model/model_final.zip')
# 
# # For tournament: upload the full checkpoints/tournament folder
# # or just re-run - it auto-resumes from saved history